# Paper 1 (brain paper): figures

Single-column figures (8.7 cm, saved at exactly that width, 400 dpi) from `paper/data/v3/` and `paper/data/p1/` (run `paper1_analysis.ipynb` first), written to `paper/figures/p1/`. Development and aging colours, and cohort colours, are v1's.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

REPO = Path.cwd().resolve().parent                       # this notebook lives in paper/
DATA, D3, D1, OUT = REPO / "paper/data", REPO / "paper/data/v3", REPO / "paper/data/p1", REPO / "paper/figures/p1"
OUT.mkdir(parents=True, exist_ok=True)
W1 = 3.42                                                # PNAS single column, 8.7 cm; figures are saved at exactly this width
INK, INK2 = "#1a1a1a", "#666666"
ORDER = ["dHCP", "BCP", "CALM", "RED", "ACE", "HCPd", "HCPya", "camCAN", "HCPa"]
COH = dict(zip(ORDER, ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#7f7f7f", "#bcbd22", "#17becf"]))  # cohort colours of v1

sns.set_theme(context="paper", style="ticks")
mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "mathtext.fontset": "stix", "font.size": 7.5, "axes.labelsize": 7.5, "axes.titlesize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 6.8, "legend.title_fontsize": 6.8,
    "axes.linewidth": 0.6, "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "text.color": INK, "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.minor.width": 0.45,
    "ytick.minor.width": 0.45, "xtick.major.size": 2.5, "ytick.major.size": 2.5, "xtick.minor.size": 1.4,
    "ytick.minor.size": 1.4, "xtick.major.pad": 1.6, "ytick.major.pad": 1.6, "axes.labelpad": 1.8,
    "lines.linewidth": 1.0, "legend.frameon": False, "legend.handlelength": 1.3, "legend.handletextpad": 0.4,
    "legend.borderaxespad": 0.2, "legend.labelspacing": 0.25, "pdf.fonttype": 42, "savefig.dpi": 400})


def letter(ax, s, x=-0.3, y=1.0):
    ax.text(x, y, s, transform=ax.transAxes, fontsize=9.5, fontweight="bold", va="bottom", ha="left")


def save(fig, name):
    """Exact figure size (no tight bounding box), so the text keeps its size in the column."""
    for ext in ("pdf", "png"):
        fig.savefig(OUT / f"{name}.{ext}", dpi=400)


def r_(a, b):
    return float(np.corrcoef(a, b)[0, 1])


def age_means(d, col):
    """Mean of col per cohort and age: 1-year bins, 3-month bins below 2 years (as in v1)."""
    a = np.where(d.age < 2, np.floor(d.age * 4) / 4 + 0.125, np.floor(d.age) + 0.5)
    return d.assign(abin=a).groupby(["cohort", "abin"])[col].mean().reset_index()


def trajectories(ax, d, col, ylabel):
    m = age_means(d, col)
    for c in ORDER:
        g = m[m.cohort == c]
        ax.plot(g.abin, g[col], "-o", color=COH[c], ms=1.7, lw=0.55, mew=0)
    ax.set(xlabel="age (years)", ylabel=ylabel, xlim=(-2, 92), xticks=[0, 20, 40, 60, 80])

DEV, AGI = "#E6A117", "#6A51A3"                         # development and aging, as in v1

In [ ]:
d = pd.read_csv(D3 / "subjects_v3.csv"); N = json.load(open(D3 / "numbers_v3.json"))      # structure, simulated MC
N1 = json.load(open(D1 / "numbers_p1.json")); race = pd.read_csv(D1 / "race_metrics.csv")
RR = pd.read_csv(D1 / "cii_r2_by_degree.csv"); nl = pd.read_csv(D1 / "nulls_sim.csv"); kap = pd.read_csv(D1 / "kappa_sim.csv")
uk = pd.read_csv(D3 / "upsilon_k.csv"); nodes_k = pd.read_csv(D3 / "nodes_sample.csv")
FX = mpl.ticker.FixedLocator; SF = mpl.ticker.ScalarFormatter

## Fig. memory (replaces v1 Table 1 and Fig. 5)
(a) $R^2$ of subgraph centrality with each factor alone, with the order-2 term and with orders 2 + 3, by degree; (b) MC against the stable rank; (c) correlation with MC by regime; (d) partial correlation given the stable rank; (e) null models.

In [ ]:
fig = plt.figure(figsize=(W1, 5.75))
gs = fig.add_gridspec(3, 2, left=0.13, right=0.985, top=0.975, bottom=0.075, hspace=0.6, wspace=0.5, height_ratios=[1, 1.3, 0.95])
R, RG = N1["race"], N1["race_regime"]

# (a) subgraph centrality: neither factor alone; their product, the order-2 term, at every degree; triangles add little
ax = fig.add_subplot(gs[0, 0]); kc = np.sqrt(RR.k_lo * RR.k_hi)
A2 = N1["cii_r2_all"]
for key, lab, col, ls, lw in (("Y", "$Y_i$", "#bdbdbd", ":", 1.2), ("s2", "$\\tilde s_i^2$", "#8c8c8c", "--", 1.0),
                              ("ord2", "$\\tilde s_i^2Y_i$ (order 2)", "#08519c", "-", 1.7), ("ord3", "+ order 3", "#e6550d", "-", 0.8)):
    ax.plot(kc, RR[f"r2_{key}"], ls=ls, color=col, lw=lw, marker="o" if key.startswith("ord") else None, ms=1.8, mew=0,
            label=f"{lab}: {A2[key]:.4f}" if key.startswith("ord") else f"{lab}: {A2[key]:.2f}")
ax.set(xscale="log", xlabel="degree $k$", ylabel="$R^2$ with $C_{ii}$", ylim=(-0.03, 1.06), yticks=[0, 0.5, 1])
ax.xaxis.set_major_locator(FX([3, 10, 30])); ax.xaxis.set_major_formatter(SF()); ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
ax.legend(loc="center left", bbox_to_anchor=(0.0, 0.5), fontsize=5.8, handlelength=1.7); letter(ax, "a")

# (b) memory against the stable rank, the network sum of the order-2 terms
ax = fig.add_subplot(gs[0, 1])
for col, m in ((DEV, d.age <= 32), (AGI, d.age > 32)):
    g = d[m]; ax.scatter(g.rs, g.MC, s=0.8, color=col, lw=0, alpha=0.4, rasterized=True)
b = np.polyfit(np.log(d.rs), d.MC, 1); xs = np.geomspace(d.rs.min(), d.rs.max(), 50); ax.plot(xs, np.polyval(b, np.log(xs)), color=INK, lw=0.9)
ax.set_xscale("log"); ax.set_xticks([3, 5, 10]); ax.xaxis.set_major_formatter(SF()); ax.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
ax.text(0.05, 0.95, f"$r$ = {R['rs']['r']:.2f}", transform=ax.transAxes, va="top")
ax.set(xlabel="stable rank $r_s$", ylabel="memory capacity", yticks=[8, 10, 12]); letter(ax, "b")

# (c) horse race as in v1, by regime; (d) what each predictor adds once the stable rank is known
pos = gs[1, :].get_position(fig); x0, wc, gap = 0.385, 0.38, 0.05; wd = 0.985 - x0 - wc - gap
axc = fig.add_axes([x0, pos.y0, wc, pos.height]); axd = fig.add_axes([x0 + wc + gap, pos.y0, wd, pos.height], sharey=axc)
items = [("stable rank $r_s$", "rs"), ("subgraph centrality", "cii"), ("triangles (order 3)", "tr3"), ("communicability", "comm"),
         ("disparity ratio", "Ratio"), ("strength", "s_raw"), ("degree", "k"), ("average controllability", "avgctrl")]
y = np.arange(len(items))[::-1]; h = 0.37
for yi, (lab, key) in zip(y, items):
    axc.barh(yi + h / 2, RG[key]["development"], height=h, color=DEV, edgecolor="white", lw=0.3)
    axc.barh(yi - h / 2, RG[key]["aging"], height=h, color=AGI, edgecolor="white", lw=0.3)
    if key != "rs":
        axd.barh(yi, R[key]["partial"], height=0.55, color="#e6550d" if key == "tr3" else "#a3a3a3", edgecolor="white", lw=0.3)
axd.text(0.02, y[0], "—", va="center", ha="left", color=INK2, fontsize=6.5)
axc.set_yticks(y); axc.set_yticklabels([i[0] for i in items], fontsize=6.4); axc.tick_params(axis="y", length=0)
for t in axc.get_yticklabels()[:1]: t.set_fontweight("bold")
axc.axvline(0, color=INK, lw=0.5); axc.set(xlabel="$r$ with memory capacity", xlim=(-0.4, 1.0), xticks=[0, 0.5, 1], ylim=(-1.6, len(items) - 0.5))
axc.set_xticklabels(["0", "0.5", "1"])
axd.axvline(0, color=INK, lw=0.5); axd.set(xlabel="given $r_s$", xlim=(-0.6, 0.45), xticks=[-0.5, 0])
axd.tick_params(axis="y", length=0, labelleft=False)
axc.legend(handles=[mpl.patches.Patch(color=DEV, label="development"), mpl.patches.Patch(color=AGI, label="aging")],
           loc="lower right", ncol=2, fontsize=6, columnspacing=0.8)
letter(axc, "c", x=-0.96); letter(axd, "d", x=-0.12)

# (e) null models (simulated, 10 realisations): observed change in memory and the change predicted by the stable rank
ax = fig.add_subplot(gs[2, :])
names = {"Uniform": "Uniform", "BrokenStick": "Broken\nstick", "Reshuffle": "Reshuffle", "EqualStrength": "Equal\nstrength", "InterHalf": "Interhem.\nweights ×½"}
NC = dict(zip(names.values(), ["#e6550d", "#fdae6b", "#969696", "#3182bd", "#756bb1"]))
q = nl.assign(null=nl.model.map(names)); lab = list(names.values()); ax.axhline(0, color=INK2, lw=0.5)
sns.boxplot(data=q, x="null", y="dMC", order=lab, color="white", width=0.42, fliersize=0, linewidth=0.6, ax=ax,
            boxprops=dict(edgecolor=INK2), whiskerprops=dict(color=INK2), medianprops=dict(color=INK), capprops=dict(color=INK2))
sns.stripplot(data=q, x="null", y="dMC", order=lab, hue="null", palette=NC, size=2, jitter=0.17, alpha=0.8, ax=ax, legend=False, rasterized=True)
pred = q.groupby("null").dMC_pred.median().reindex(lab)
ax.scatter(np.arange(len(lab)), pred.values, marker="D", s=18, color=INK, edgecolor="white", lw=0.5, zorder=5, label="predicted from $r_s$")
ax.set(xlabel="", ylabel="$\\Delta$MC (null $-$ real)"); ax.tick_params(axis="x", length=0); ax.legend(loc="upper left"); letter(ax, "e", x=-0.135)
for a in fig.axes:
    sns.despine(ax=a)
sns.despine(ax=axd, left=True)
save(fig, "fig_memory")

## Fig. lifespan and aging (replaces v1 Fig. 4)
(a) MC across the lifespan by cohort; (b) aging within camCAN and HCPa, z-scores: MC and $C_{ii}$ rise, disparity flat, $h$ falls; (c) correlations with age within cohorts, and MC given $h$; (d) virtual aging.

In [ ]:
fig = plt.figure(figsize=(W1, 4.55))
gs = fig.add_gridspec(2, 2, left=0.13, right=0.985, top=0.975, bottom=0.165, hspace=0.55, wspace=0.55)
W = N1["within"]; VR = N1["virtual_rescaling"]
# (a) memory across the lifespan: per-age means of each cohort
ax = fig.add_subplot(gs[0, 0]); trajectories(ax, d, "MC", "memory capacity"); ax.set_yticks([9, 10, 11]); letter(ax, "a")
# (b) aging within cohorts: memory and subgraph centrality rise, disparity stays flat, interhemispheric weight falls
ax = fig.add_subplot(gs[0, 1])
a = d[d.cohort.isin(["camCAN", "HCPa"]) & (d.age >= 36)].merge(race[["sid", "cii"]], on="sid")
z = a.copy()
for col in ("MC", "cii", "Ratio", "hom_F2"):
    z[col] = a.groupby("cohort")[col].transform(lambda v: (v - v.mean()) / v.std())
g = z.groupby(pd.cut(z.age, [36, 45, 55, 65, 75, 91]), observed=True)
LC = {"Ratio": ("disparity ratio", "#9e9e9e"), "cii": ("$C_{ii}$", "#e6550d"), "MC": ("memory capacity", INK), "hom_F2": ("homotopic share $h$", "#6a3d9a")}
for col, (lab, c) in LC.items():
    ax.errorbar(g.age.mean(), g[col].mean(), yerr=1.96 * g[col].sem(), color=c, marker="o", ms=2.4, mew=0, lw=1.1, capsize=1.2, label=lab)
ax.axhline(0, color=INK2, lw=0.5)
ax.set(xlabel="age (years)", ylabel="z-score within cohort", xticks=[40, 60, 80], ylim=(-1.55, 0.9))
ax.legend(loc="lower left", fontsize=5.8); letter(ax, "b")
# (c) correlation with age within camCAN and HCPa; once h is controlled, age no longer predicts memory
ax = fig.add_subplot(gs[1, 0]); w = 0.16
keys = (("Ratio", "disparity", "#9e9e9e"), ("cii", "$C_{ii}$", "#e6550d"), ("MC", "MC", INK), ("hom_F2", "$h$", "#6a3d9a"), ("MC_given_h", "MC | $h$", "#d9d9d9"))
for j, (key, lab, col) in enumerate(keys):
    ax.bar(np.arange(2) + (j - 2) * w, [W[f"{c}|{key}"]["r"] for c in ("camCAN", "HCPa")], width=w * 0.92, color=col, edgecolor=INK, lw=0.3, label=lab)
ax.axhline(0, color=INK, lw=0.5); ax.set_xticks([0, 1]); ax.set_xticklabels(["camCAN", "HCPa"]); ax.tick_params(axis="x", length=0)
for t, c in zip(ax.get_xticklabels(), ("camCAN", "HCPa")):
    t.set_color(COH[c]); t.set_fontweight("bold")
ax.set(ylabel="$r$ with age", ylim=(-0.72, 0.85), xlim=(-0.5, 1.5), yticks=[-0.5, 0, 0.5])
ax.legend(loc="upper center", ncol=3, fontsize=5.6, handlelength=0.8, columnspacing=0.5, handletextpad=0.25); letter(ax, "c")
# (d) virtual aging: young connectomes with rescaled interhemispheric weights (simulated)
ax = fig.add_subplot(gs[1, 1])
for c in ("camCAN", "HCPa"):
    gk = kap[kap.cohort == c].groupby("kappa").MC.agg(["mean", "sem"])
    ax.fill_between(gk.index, gk["mean"] - 1.96 * gk["sem"], gk["mean"] + 1.96 * gk["sem"], color=COH[c], alpha=0.25, lw=0)
    ax.plot(gk.index, gk["mean"], "-o", color=COH[c], ms=2, mew=0, lw=1)
    ax.axhline(VR[c]["MC_old"], color=COH[c], lw=0.8, ls="--")
    Fy, Fo = VR[c]["inter_young"], VR[c]["inter_old"]
    ax.axvline(Fo * (1 - Fy) / (Fy * (1 - Fo)), color=COH[c], lw=0.7, ls=":")
ax.text(1.52, VR["camCAN"]["MC_old"] + 0.03, "≥ 70 y, observed", ha="right", va="bottom", fontsize=6, color=INK2)
ax.set(xlabel="interhem. weights × $\\kappa$", ylabel="MC, connectomes < 40 y", xticks=[0.5, 1, 1.5]); letter(ax, "d")
hs = [mpl.lines.Line2D([], [], color=COH[c], marker="o", ms=3, lw=0.7, label=c) for c in ORDER]
fig.legend(handles=hs, loc="lower center", bbox_to_anchor=(0.55, 0.0), ncol=5, columnspacing=0.8, handletextpad=0.3, fontsize=6.6)
for a_ in fig.axes:
    sns.despine(ax=a_)
save(fig, "fig_lifespan_aging")

## SI: full lifespan trajectories (disparity vs degree, disparity ratio, MC, stable rank, $h$; MC vs $h$; within-cohort aging; virtual aging)

In [ ]:
fig = plt.figure(figsize=(W1, 6.1))
outer = fig.add_gridspec(1, 2, left=0.135, right=0.985, top=0.975, bottom=0.112, wspace=0.52)
gl = outer[0].subgridspec(2, 1, height_ratios=[1.25, 3.75], hspace=0.16)
gla = gl[1].subgridspec(4, 1, hspace=0.3)
gr = outer[1].subgridspec(3, 1, height_ratios=[1.3, 1.15, 1.3], hspace=0.34)

# (a) nodal disparity against degree (v1 Fig. 4a)
ax = fig.add_subplot(gl[0]); ax_a = ax
ax.scatter(nodes_k.k, nodes_k.Ups, s=0.5, color="#6baed6", alpha=0.18, lw=0, rasterized=True)
k = np.arange(1, 90.0); mu = 2 * k / (k + 1)
sd = k * np.sqrt((20 + 4 * k) / ((k + 1) * (k + 2) * (k + 3)) - 4 / (k + 1) ** 2)
ax.fill_between(k, np.clip(mu - 2 * sd, 0.7, None), mu + 2 * sd, color="#bdbdbd", alpha=0.8, lw=0, label="null $\\pm2\\sigma$")
ax.plot(k, mu, color="#636363", lw=0.7)
ax.plot(k, k, color="#d62728", lw=0.8, ls="--", label="$\\Upsilon=k$")
g = uk[uk["size"] >= 30]
ax.plot(g.k, g["mean"], color="#08519c", lw=1.3, label=f"$\\langle\\Upsilon\\rangle_k\\propto k^{{{N['upsilon_alpha']:.2f}}}$")
ax.set(xscale="log", yscale="log", xlabel="degree $k$", ylabel="$\\Upsilon=kY$", xlim=(1, 90), ylim=(0.7, 90))
for axis in (ax.xaxis, ax.yaxis):
    axis.set_major_locator(mpl.ticker.FixedLocator([1, 3, 10, 30, 90])); axis.set_major_formatter(mpl.ticker.ScalarFormatter())
    axis.set_minor_locator(mpl.ticker.NullLocator())
ax.legend(loc="upper left", fontsize=6.3); letter(ax, "a")

# (b-e) lifespan trajectories on a shared age axis: per-age means of each cohort
spec = (("Ratio", "$\\langle\\Upsilon/\\Upsilon_{null}\\rangle$"), ("MC", "MC"), ("rs", "$r_s$"), ("hom_F2", "$h$"))
axs = []
for i, (col, lab) in enumerate(spec):
    ax = fig.add_subplot(gla[i], sharex=axs[0] if axs else None); trajectories(ax, d, col, lab)
    if i < 3:
        ax.set_xlabel(""); ax.tick_params(labelbottom=False)
    letter(ax, "bcde"[i]); axs.append(ax)
fig.align_ylabels([ax_a] + axs)

# (f) memory against the homotopic share, coloured by age
ax = fig.add_subplot(gr[0])
sc = ax.scatter(d.hom_F2, d.MC, c=d.age, cmap="viridis", vmin=0, vmax=90, s=0.8, lw=0, alpha=0.65, rasterized=True)
cax = ax.inset_axes([0.5, 0.9, 0.46, 0.05]); cb = fig.colorbar(sc, cax=cax, orientation="horizontal")
cb.set_ticks([0, 45, 90]); cb.ax.tick_params(labelsize=6, length=1.5, pad=1, width=0.4); cb.outline.set_linewidth(0.3)
cax.set_title("age (years)", fontsize=6.3, pad=1.5)
ax.text(0.05, 0.05, f"$r$ = {r_(d.hom_F2, d.MC):.2f}", transform=ax.transAxes)
ax.set(xlabel="homotopic share $h$", ylabel="MC", ylim=(d.MC.min() - 0.2, d.MC.max() + 1.1), yticks=[8, 10, 12])
letter(ax, "f", x=-0.36)

# (g) within camCAN and HCPa: the homotopic share accounts for the aging rise of memory
ax = fig.add_subplot(gr[1]); L = N["lifespan"]; w = 0.27
bars = (("r_age_hom", "$h$", "#6a3d9a"), ("r_age_MC", "MC", INK), ("pr_age_MC_given_hom", "MC | $h$", "#bdbdbd"))
for j, (key, lab, col) in enumerate(bars):
    ax.bar(np.arange(2) + (j - 1) * w, [L[c][key] for c in ("camCAN", "HCPa")], width=w * 0.92, color=col, edgecolor=INK,
           lw=0.3, label=lab)
ax.axhline(0, color=INK, lw=0.5); ax.set_xticks([0, 1]); ax.set_xticklabels(["camCAN", "HCPa"]); ax.tick_params(axis="x", length=0)
for t, c in zip(ax.get_xticklabels(), ("camCAN", "HCPa")):
    t.set_color(COH[c]); t.set_fontweight("bold")
ax.set(ylabel="$r$ with age", ylim=(-0.72, 0.62), xlim=(-0.5, 1.5), yticks=[-0.5, 0, 0.5])
ax.legend(loc="upper center", ncol=3, handlelength=0.8, columnspacing=0.6, handletextpad=0.25); letter(ax, "g", x=-0.36)

# (h) virtual aging: young connectomes with rescaled interhemispheric weights
ax = fig.add_subplot(gr[2]); VR = N1["virtual_rescaling"]
for c in ("camCAN", "HCPa"):
    g = kap[kap.cohort == c].groupby("kappa").MC.agg(["mean", "sem"])
    ax.fill_between(g.index, g["mean"] - 1.96 * g["sem"], g["mean"] + 1.96 * g["sem"], color=COH[c], alpha=0.25, lw=0)
    ax.plot(g.index, g["mean"], "-o", color=COH[c], ms=2, mew=0, lw=1)
    ax.axhline(VR[c]["MC_old"], color=COH[c], lw=0.8, ls="--")
    Fy, Fo = VR[c]["inter_young"], VR[c]["inter_old"]                # interhemispheric fraction of the total weight
    ax.axvline(Fo * (1 - Fy) / (Fy * (1 - Fo)), color=COH[c], lw=0.7, ls=":")   # kappa that brings Fy to Fo
ax.text(1.52, VR["camCAN"]["MC_old"] + 0.03, "≥ 70 y, observed", ha="right", va="bottom", fontsize=6.3, color=INK2)
ax.text(0.47, 9.8, "$\\kappa$ matching\n≥ 70 y", ha="left", va="bottom", fontsize=6.3, color=INK2)
ax.set(xlabel="interhem. weights × $\\kappa$", ylabel="MC", xticks=[0.5, 1, 1.5])
letter(ax, "h", x=-0.36)

hs = [mpl.lines.Line2D([], [], color=COH[c], marker="o", ms=3, lw=0.7, label=c) for c in ORDER]
fig.legend(handles=hs, loc="lower center", bbox_to_anchor=(0.54, 0.0), ncol=5, columnspacing=0.8, handletextpad=0.3, fontsize=6.8)
for a in fig.axes:
    if a is not cax:
        sns.despine(ax=a)
save(fig, "figS_lifespan_full")

## SI: development, within cohorts

In [ ]:
fig = plt.figure(figsize=(W1, 4.3))
gs = fig.add_gridspec(2, 2, left=0.16, right=0.985, top=0.97, bottom=0.09, hspace=0.5, wspace=0.55, height_ratios=[1.15, 1])
W = N1["within"]; coh = [c for c in ORDER if f"{c}|MC" in W]
# (a) age trends within each cohort (95% CI)
ax = fig.add_subplot(gs[0, 0]); y = np.arange(len(coh))[::-1]
for off, col, key, lab in ((0.16, INK, "MC", "MC"), (-0.16, "#6a3d9a", "hom_F2", "$h$")):
    for yi, c in zip(y, coh):
        w = W[f"{c}|{key}"]; ax.plot([w["lo"], w["hi"]], [yi + off] * 2, color=col, lw=0.8); ax.plot(w["r"], yi + off, "o", color=col, ms=2.6)
    ax.plot([], [], "o-", color=col, ms=2.6, lw=0.8, label=lab)
ax.axvline(0, color=INK2, lw=0.5); ax.set_yticks(y); ax.set_yticklabels(coh, fontsize=6.2); ax.tick_params(axis="y", length=0)
for t, c in zip(ax.get_yticklabels(), coh): t.set_color(COH[c])
ax.set(xlabel="correlation with age\nwithin the cohort", xlim=(-0.75, 0.75)); ax.legend(loc="upper right"); letter(ax, "a", x=-0.45)
# (b) the turning point of memory within camCAN (single site)
ax = fig.add_subplot(gs[0, 1]); cc = d[d.cohort == "camCAN"]; tp = N1["camcan_turning_point"]["MC"]
ax.scatter(cc.age, cc.MC, s=1.2, color=COH["camCAN"], alpha=0.45, lw=0, rasterized=True)
bm = cc.groupby(pd.cut(cc.age, np.arange(15, 95, 5)), observed=True).agg(a=("age", "mean"), m=("MC", "mean"), s=("MC", "sem"))
ax.errorbar(bm.a, bm.m, yerr=1.96 * bm.s, color=INK, lw=0.9, marker="o", ms=2, capsize=1)
ax.axvspan(*tp["ci"], color="#d9d9d9", alpha=0.6, lw=0); ax.axvline(tp["age"], color=INK, lw=0.7, ls="--")
ax.text(tp["age"] + 1, ax.get_ylim()[1], f"{tp['age']:.0f} y", va="top", fontsize=6.3)
ax.set(xlabel="age (years)", ylabel="MC (camCAN)", xticks=[20, 40, 60, 80]); letter(ax, "b")
# (c) young adults of two cohorts at the same ages
for j, (col, lab) in enumerate((("MC", "memory capacity"), ("hom_F2", "homotopic share $h$"))):
    ax = fig.add_subplot(gs[1, j]); m = d[(d.age >= 22) & (d.age <= 37) & d.cohort.isin(["HCPya", "camCAN"])]
    sns.boxplot(data=m, x="cohort", y=col, order=["HCPya", "camCAN"], color="white", width=0.45, fliersize=0, linewidth=0.6, ax=ax,
                boxprops=dict(edgecolor=INK2), whiskerprops=dict(color=INK2), medianprops=dict(color=INK), capprops=dict(color=INK2))
    sns.stripplot(data=m, x="cohort", y=col, order=["HCPya", "camCAN"], hue="cohort", palette=COH, size=1.4, jitter=0.18, alpha=0.6, ax=ax, legend=False, rasterized=True)
    ax.set(xlabel="ages 22-37", ylabel=lab); ax.tick_params(axis="x", length=0); letter(ax, "c" if j == 0 else "d", x=-0.38)
for a in fig.axes:
    sns.despine(ax=a)
save(fig, "figS_development")